# C2Align: ADRD prediction

This notebook contains the task-specific Specialist and Parallel branch prompts, their inference workflow, and Method 5 MoE_b (Dual KL Calib).


In [ ]:
import gc
import json
import os
import random
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, brier_score_loss, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.auto import tqdm

for _repository_root in (Path.cwd(), *Path.cwd().parents):
    if (_repository_root / "c2align_feature_policy.py").is_file():
        sys.path.insert(0, str(_repository_root))
        break
else:
    raise RuntimeError("Could not locate c2align_feature_policy.py.")
from c2align_feature_policy import numeric_feature_matrix

GLOBAL_SEED = 1
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)


## Local inference callback


In [ ]:
INFERENCE_CALLBACK = None


def configure_inference_callback(callback):
    global INFERENCE_CALLBACK
    INFERENCE_CALLBACK = callback


def call_llm(prompt, system_prompt=None, seed=GLOBAL_SEED, temperature=0.0):
    if INFERENCE_CALLBACK is None:
        raise RuntimeError(
            "Configure a local inference callback or load previously generated branch outputs."
        )
    raw = INFERENCE_CALLBACK(
        system_prompt=system_prompt,
        prompt=prompt,
        seed=seed,
        temperature=temperature,
    )
    if isinstance(raw, str):
        raw = raw.strip().removeprefix("```json").removesuffix("```").strip()
        raw = json.loads(raw)
    elif isinstance(raw, dict) and "response" in raw:
        raw = raw["response"]
    if not isinstance(raw, dict):
        raise TypeError("The inference callback must return a JSON object or a JSON string.")
    return {"response": raw, "success": True}


def _safe_call(prompt, system_prompt, seed=GLOBAL_SEED):
    try:
        return call_llm(prompt, system_prompt=system_prompt, seed=seed)
    except Exception:
        return {"response": {}, "success": False}


## Specialist branch and complete prompts


In [ ]:
SPECIALIST_PROMPTS = {'specialist_symptom': {'system': "You are a 'Symptom Specialist' for Alzheimer's Disease and "
                                  'Related Dementias (ADRD).\n'
                                  'Analyze the clinical text provided.\n'
                                  'TASK: Extract specific symptoms and signs related to cognitive, '
                                  'behavioral, and physical decline. Return JSON only.\n'
                                  '\n'
                                  'TARGET SYMPTOMS & SIGNS:\n'
                                  '1. **Cognitive:** [Memory loss, Amnesia, Confusion, '
                                  'Disorientation, Executive dysfunction, Aphasia, Agnosia, '
                                  'Apraxia].\n'
                                  '2. **Behavioral:** [Agitation, Aggression, Wandering, '
                                  'Hallucinations (Visual/Auditory), Apathy, Disinhibition, '
                                  'Personality changes].\n'
                                  '3. **Physical/Neuro:** [Gait disturbance, Falls, Tremor, '
                                  'Rigidity, Bradykinesia, Focal neurological signs (hemiparesis), '
                                  'Incontinence].\n'
                                  '4. **General Context:** "Progressive decline", "Cognitive '
                                  'impairment", "Dementia", "Senile".\n'
                                  '\n'
                                  'OUTPUT JSON FORMAT:\n'
                                  '{\n'
                                  '"symptoms_list": ["list of detected '
                                  'cognitive/behavioral/physical symptoms found"],\n'
                                  '"has_behavioral_disturbance": <bool>,\n'
                                  '"has_gait_or_falls": <bool>,\n'
                                  '"has_memory_complaint": <bool>\n'
                                  '}'},
 'specialist_pharma': {'system': "You are a 'Pharmacologist Agent' for ADRD.\n"
                                 'Analyze the clinical text provided.\n'
                                 'TASK: Extract medications specifically used for Dementia or its '
                                 'behavioral management. Return JSON only.\n'
                                 'TARGET CLASSES:\n'
                                 '1. **Cholinesterase Inhibitors:** [Donepezil (Aricept), '
                                 'Rivastigmine (Exelon), Galantamine (Razadyne)].\n'
                                 '2. **NMDA Antagonists:** [Memantine (Namenda)].\n'
                                 '3. **Behavioral Mgmt (Supportive):** [Risperidone, Quetiapine, '
                                 'Olanzapine, SSRIs (Sertraline, Citalopram)].\n'
                                 '\n'
                                 'OUTPUT JSON FORMAT:\n'
                                 '{\n'
                                 '"medications": [\n'
                                 '{\n'
                                 '"class": "Cholinesterase Inhibitor",\n'
                                 '"name": "Donepezil",\n'
                                 '"reason": "for memory loss"\n'
                                 '}\n'
                                 ']\n'
                                 '}'},
 'specialist_risk': {'system': "You are a 'Diagnostic Test & Risk Factor Specialist' for ADRD.\n"
                               'Analyze the clinical text provided.\n'
                               'TASK: Extract Tests, Imaging findings, Cognitive Scores, and Risk '
                               'Factors. Return JSON only.\n'
                               '1. **TESTS & IMAGING**: \n'
                               '   - **Imaging:** [MRI Brain, CT Head, PET Scan].\n'
                               '   - **Key Findings:** [Atrophy (Hippocampal/Temporal/Cortical), '
                               'White matter changes (Leukoaraiosis), Ischemic changes, Infarcts, '
                               'Lacunes, Ventriculomegaly, Hydrocephalus].\n'
                               '   - **Cognitive Scores:** Extract score strings for MMSE, MoCA, '
                               'Mini-Cog (e.g., "MMSE 18/30", "MoCA 22").\n'
                               '2. **RISK FACTORS / HISTORY**: \n'
                               '   - [Stroke, TIA, Cerebrovascular disease, Hypertension, '
                               'Diabetes, Family history of dementia, Head trauma].\n'
                               '\n'
                               'OUTPUT JSON FORMAT:\n'
                               '{\n'
                               '"imaging_findings": ["list of specific imaging findings"],\n'
                               '"mmse_extracted": "string or \'na\'",\n'
                               '"moca_extracted": "string or \'na\'",\n'
                               '"risks_history": ["list of risk factors found"]\n'
                               '}'},
 'agent1_adrd': {'system': 'You are a Senior Expert Neurologist and Geriatrician.\n'
                           "Your Goal: Determine if the patient has Alzheimer's Disease and "
                           'Related Dementias (ADRD) based on the Specialist Reports provided.\n'
                           'Input: Summarized findings from Symptom, Pharmacologist, and Risk '
                           'Specialists.\n'
                           '\n'
                           'Criteria:\n'
                           '* **POSITIVE (1)**: Definitive diagnosis code OR Clear clinical '
                           'description of dementia/cognitive decline + treatment (e.g. Donepezil) '
                           'OR Supportive imaging (Atrophy).\n'
                           '* **NEGATIVE (0)**: No mention of dementia. Symptoms explained by '
                           'other causes (e.g., Delirium, Depression, Normal pressure '
                           'hydrocephalus).\n'
                           '\n'
                           'WARNING - CONFIDENCE SCORING:\n'
                           'For `adrd_confidence`, you MUST output the PROBABILITY THAT THE '
                           'PATIENT HAS ADRD.\n'
                           '* 0 - 49: You believe the patient DOES NOT have ADRD. (e.g., 0 means '
                           'you are 100% certain they are healthy).\n'
                           '* 50 - 100: You believe the patient HAS ADRD. (e.g., 100 means you are '
                           '100% certain they have ADRD).\n'
                           'Do NOT output a high number just because you are "confident they do '
                           'not have it". High numbers ALWAYS mean "High probability of having the '
                           'disease".\n'
                           '\n'
                           'Output JSON format:\n'
                           '{\n'
                           '    "has_adrd": <bool>,\n'
                           '    "adrd_confidence": <0-100 integer representing the probability of '
                           'HAVING ADRD (0=definitely healthy, 100=definitely diseased)>,\n'
                           '    "reason_adrd": "<Brief medical reasoning citing specific symptoms '
                           'or tests (max 40 words)>"\n'
                           '}\n'
                           'Do NOT default to Yes unless evidence supports it.'},
 'agent3_subtype': {'system': 'You are an ADRD Phenotyping Specialist.\n'
                              'Your Goal: Classify the ADRD subtype based on the Diagnosis and '
                              'Specialist evidence.\n'
                              'Input: Diagnostic Agent result and Specialist results.\n'
                              '\n'
                              'Rules:\n'
                              '- **IF `has_adrd` is False**: The subtype MUST be **"na"**.\n'
                              '- **IF `has_adrd` is True**: Assign ONE primary subtype:\n'
                              "    * **ad**: Alzheimer's Disease (Hippocampal atrophy, Memory "
                              'predominant).\n'
                              '    * **vd**: Vascular Dementia (Multi-infarct, Post-stroke, '
                              'Stepwise decline, White matter changes).\n'
                              '    * **fd**: Frontotemporal Dementia (Behavioral variant, Language '
                              'variant/PPA).\n'
                              '    * **nsd**: Non-specific / Mixed / Other Dementias.\n'
                              '\n'
                              'Output JSON format:\n'
                              '{\n'
                              '    "adrd_subtype": "ad" | "vd" | "fd" | "nsd" | "na",\n'
                              '    "subtype_probabilities": {\n'
                              '        "na": <0-100 integer>,\n'
                              '        "ad": <0-100 integer>,\n'
                              '        "vd": <0-100 integer>,\n'
                              '        "fd": <0-100 integer>,\n'
                              '        "nsd": <0-100 integer>\n'
                              '    },\n'
                              '    "reason_subtype": "<Brief reasoning based on specific evidence '
                              '(max 40 words)>"\n'
                              '}    \n'
                              "Note: Ensure the 5 probability values in 'subtype_probabilities' "
                              'sum to exactly 100.'},
 'agent4_chart_review': {'system': 'You are a Senior ADRD Diagnostic Specialist specializing in '
                                   'Chart Review.\n'
                                   'Your Goal: Review the findings of the previous agents '
                                   '(Specialists + Diagnosis) against the original patient data.\n'
                                   'Correct any hallucinations or logic errors.\n'
                                   '\n'
                                   'Input will include:\n'
                                   '1. Original Clinical Note\n'
                                   '2. Specialist Findings (Symptoms, Meds, Risks)\n'
                                   '3. Preliminary Diagnosis & Subtype\n'
                                   '\n'
                                   'Output JSON format (Final Validated Result):\n'
                                   '{\n'
                                   '    "has_adrd": <bool>,\n'
                                   '    "adrd_confidence": <0-100 integer representing the '
                                   'probability of HAVING ADRD (0=definitely healthy, '
                                   '100=definitely diseased)>,\n'
                                   '    "mmse_value_extracted": "string or \'na\'", \n'
                                   '    "adrd_subtype": "<subtype string>",\n'
                                   '    "subtype_probabilities": {\n'
                                   '        "na": <0-100 integer>,\n'
                                   '        "ad": <0-100 integer>,\n'
                                   '        "vd": <0-100 integer>,\n'
                                   '        "fd": <0-100 integer>,\n'
                                   '        "nsd": <0-100 integer>\n'
                                   '    },\n'
                                   '    "audit_reason": "<Why you kept or changed the decision>"\n'
                                   '}\n'
                                   '\n'
                                   'WARNING - STRICT JSON SEMANTICS:\n'
                                   "1. Your output JSON MUST represent the PATIENT'S actual "
                                   'condition, NOT your agreement with the previous AI. \n'
                                   '   - "has_adrd": 1 (or true) means the PATIENT HAS ADRD.\n'
                                   '   - "has_adrd": 0 (or false) means the PATIENT DOES NOT HAVE '
                                   'ADRD.\n'
                                   '   DO NOT output "true" just to mean "I agree with the '
                                   'previous agent".\n'
                                   '2. "adrd_confidence" MUST represent the probability of HAVING '
                                   'the disease. If you confidently agree the patient is healthy, '
                                   'this value must be close to 0.\n'
                                   '\n'
                                   'Note: Ensure the 5 probability values in '
                                   "'subtype_probabilities' sum to exactly 100."}}


def flatten_specialist_output(name, data):
    features = {}
    for key, value in data.items():
        feature_name = f"feat_{name}_{key}"
        if isinstance(value, bool):
            features[feature_name] = int(value)
        elif isinstance(value, (int, float)) and not isinstance(value, bool):
            features[feature_name] = float(value)
        elif isinstance(value, list):
            features[f"{feature_name}_count"] = len(value)
        elif isinstance(value, dict):
            for child_key, child_value in value.items():
                if isinstance(child_value, (bool, int, float)):
                    features[f"{feature_name}_{child_key}"] = float(child_value)
    return features


def run_specialist_branch(row, use_chart_review=True):
    note = str(row.get("text", ""))
    context = (
        f"Age: {row.get('anchor_age')}, Gender: {row.get('gender')}, "
        f"Diagnoses: {row.get('all_icd_codes')}"
    )
    specialist_keys = ["specialist_symptom", "specialist_pharma", "specialist_risk"]
    with ThreadPoolExecutor(max_workers=3) as executor:
        futures = [
            executor.submit(
                _safe_call,
                note,
                SPECIALIST_PROMPTS[key]["system"],
                GLOBAL_SEED + offset,
            )
            for offset, key in enumerate(specialist_keys)
        ]
    specialist_results = [future.result() for future in futures]
    specialist_data = [
        result["response"] if result["success"] else {}
        for result in specialist_results
    ]
    symptom_data, medication_data, risk_data = specialist_data
    specialist_summary = json.dumps(
        {
            "symptom_specialist": symptom_data,
            "medication_specialist": medication_data,
            "risk_specialist": risk_data,
        },
        ensure_ascii=True,
    )

    diagnosis_prompt = (
        f"Patient context: {context}\nSpecialist findings: {specialist_summary}"
    )
    diagnosis_result = _safe_call(
        diagnosis_prompt,
        SPECIALIST_PROMPTS["agent1_adrd"]["system"],
    )
    diagnosis = diagnosis_result["response"] if diagnosis_result["success"] else {}

    subtype_prompt = (
        f"Diagnosis: {json.dumps(diagnosis, ensure_ascii=True)}\n"
        f"Extracted measurement: {risk_data.get('mmse_extracted', 'na')}\n"
        f"Specialist findings: {specialist_summary}"
    )
    subtype_result = _safe_call(
        subtype_prompt,
        SPECIALIST_PROMPTS["agent3_subtype"]["system"],
    )
    subtype = subtype_result["response"] if subtype_result["success"] else {}
    combined = {**diagnosis, **subtype}
    combined["mmse_value_extracted"] = risk_data.get("mmse_extracted", "na")
    combined.update(flatten_specialist_output("symptom", symptom_data))
    combined.update(flatten_specialist_output("medication", medication_data))
    combined.update(flatten_specialist_output("risk", risk_data))

    if use_chart_review:
        review_prompt = (
            f"Patient context: {context}\nOriginal note: {note}\n"
            f"Specialist findings: {specialist_summary}\n"
            f"Preliminary result: {json.dumps(combined, ensure_ascii=True)}"
        )
        review_result = _safe_call(
            review_prompt,
            SPECIALIST_PROMPTS["agent4_chart_review"]["system"],
        )
        if review_result["success"]:
            combined.update(review_result["response"])

    raw_probability = float(combined.get("adrd_confidence", 0.0))
    combined["mas_adrd_conf"] = raw_probability
    combined["mas_has_adrd"] = int(bool(combined.get("has_adrd", False)))
    combined["mas_adrd_subtype"] = str(combined.get("adrd_subtype", "na"))
    return combined


## Parallel branch and complete prompt


In [ ]:
PARALLEL_SYSTEM_PROMPT = 'You are a Senior Expert Neurologist and Geriatrician. Analyze the [Clinical Data] provided for Alzheimer\'s Disease and Related Dementias (ADRD) diagnosis.\n\n### 1. DIAGNOSTIC CHECKLIST & KEYWORDS (Strictly Scan for These):\nUse the following lists as your primary extraction targets.\n\n**A. SYMPTOMS & SIGNS:**\n* **Cognitive:** [Memory loss, Amnesia, Confusion, Disorientation, Executive dysfunction, Aphasia, Agnosia, Apraxia].\n* **Behavioral:** [Agitation, Aggression, Wandering, Hallucinations (Visual/Auditory), Apathy, Disinhibition, Personality changes].\n* **Physical/Neuro:** [Gait disturbance, Falls, Tremor, Rigidity, Bradykinesia, Focal neurological signs (hemiparesis), Incontinence].\n* **Contextual:** "Progressive decline", "Cognitive impairment", "Dementia", "Senile".\n\n**B. TESTS & IMAGING:**\n* **Imaging:** [MRI Brain, CT Head, PET Scan].\n* **Key Findings:** [Atrophy (Hippocampal/Temporal/Cortical), White matter changes (Leukoaraiosis), Ischemic changes, Infarcts, Lacunes, Ventriculomegaly, Hydrocephalus].\n* **Cognitive Scores:** specifically check for scores like MMSE, MoCA, Mini-Cog (e.g., "MMSE 18/30", "Score < 24").\n\n**C. MEDICATIONS (Dementia-Specific):**\n* **Cholinesterase Inhibitors:** [Donepezil (Aricept), Rivastigmine (Exelon), Galantamine (Razadyne)].\n* **NMDA Antagonists:** [Memantine (Namenda)].\n* **Behavioral Mgmt (Supportive):** [Risperidone, Quetiapine, Olanzapine, SSRIs].\n\n**D. RISK FACTORS & HISTORY:**\n* **History:** [Stroke, TIA, Cerebrovascular disease, Hypertension, Diabetes, Family history of dementia, Head trauma].\n\n---\n\n### 2. INSTRUCTIONS:\n\n**Step 1: Feature Assessment (Independent)**\nFor `has_adrd` field, determine if that specific data source provides evidence for ADRD.\n* **Judgment**: Assign **1** (Supports ADRD), **0** (Does not support/Normal/Irrelevant).\n* **Reasoning (`reason_adrd`)**: \n    * **MUST be < 40 words.**\n    * **If 1**: Quote the specific keyword or value found.\n    * **If 0**: State "No ADRD evidence" or explain the negative finding.\n\n**Step 2: Cognitive Score Structured Extraction (CRITICAL)**\nDo not simplify; preserve the full topology of the data. Extract MMSE/MoCA into 1 field:\n**mmse_value_extracted**: The exact string found (e.g., "18/30", "22", "<24", "Normal"). If not found, use "na".\n\n**Step 3: Holistic Synthesis (The "Purity" Filter)**\nDetermine `has_adrd` based on the convergence of evidence.\n* **1 (Confident POSITIVE)**: Definitive diagnosis code OR Clear clinical description of dementia + treatment.\n* **0 (Confident NEGATIVE)**: No mention of dementia. Symptoms explained by other causes.\n\n**Step 4: Confidence Scoring (CRITICAL SEMANTICS)**\nFor the `adrd_confidence` field, you MUST output the PROBABILITY THAT THE PATIENT HAS ADRD.\n* **0 - 49**: You believe the patient DOES NOT have ADRD. (e.g., 0 means you are 100% certain they are perfectly healthy / have no ADRD).\n* **50 - 100**: You believe the patient HAS ADRD. (e.g., 100 means you are 100% certain they have ADRD).\nWARNING: Do NOT output a high number just because you are "confident they do not have it". High numbers ALWAYS mean "High probability of having the disease".\n\n**Step 5: Phenotyping (Subtype Logic)**\nFor `adrd_subtype` field, determine the specific subtype based on evidence.\n* **IF `has_adrd` is 0**: The subtype MUST be **"na"**.\n* **IF `has_adrd` is 1**: Assign ONE primary subtype:\n    * **ad**: Alzheimer\'s Disease (Hippocampal atrophy, Memory predominant).\n    * **vd**: Vascular Dementia (Multi-infarct, Post-stroke, Stepwise decline).\n    * **fd**: Frontotemporal Dementia (Behavioral variant, Language variant/PPA).\n    * **nsd**: Non-specific / Mixed / Other Dementias.\n* **Reasoning (`reason_subtype`)**: \n    * **MUST be < 40 words.**\n\n---\n\n### 3. JSON RESPONSE FORMAT (Strict Adherence Required):\nReturn ONLY a JSON object:\n{\n  "has_adrd": <bool>,\n  "adrd_confidence": <0-100 integer representing the probability of HAVING ADRD (0=definitely healthy, 100=definitely diseased)>,\n  "adrd_subtype": "ad" | "vd" | "fd" | "nsd" | "na",\n  "subtype_probabilities": {\n      "na": <0-100 integer>,\n      "ad": <0-100 integer>,\n      "vd": <0-100 integer>,\n      "fd": <0-100 integer>,\n      "nsd": <0-100 integer>\n  },\n  "mmse_value_extracted": "string or \'na\'", \n  "reason_adrd": "<reasoning for ADRD diagnosis>",               \n  "reason_subtype": "<reasoning for subtype classification>"\n}\nNote: Ensure the 5 probability values in \'subtype_probabilities\' sum to exactly 100.\n'


def run_parallel_branch(row, num_agents=2):
    note = str(row.get("text", ""))
    context = (
        f"Age: {row.get('anchor_age')}, Gender: {row.get('gender')}, "
        f"Diagnoses: {row.get('all_icd_codes')}, Clinical note: {note}"
    )
    with ThreadPoolExecutor(max_workers=num_agents) as executor:
        futures = [
            executor.submit(
                _safe_call,
                context,
                PARALLEL_SYSTEM_PROMPT,
                GLOBAL_SEED + offset * 123,
            )
            for offset in range(num_agents)
        ]
    responses = [
        result["response"]
        for result in (future.result() for future in futures)
        if result["success"]
    ]
    if not responses:
        return {
            "mas_has_adrd": 0,
            "mas_adrd_conf": 50.0,
            "mas_adrd_subtype": "na",
        }

    probabilities = [float(item.get("adrd_confidence", 50.0)) for item in responses]
    scale = 100.0 if max(probabilities) > 1.0 else 1.0
    mean_probability = float(np.mean(probabilities))
    votes = [int(bool(item.get("has_adrd", False))) for item in responses]
    subtypes = [str(item.get("adrd_subtype", "na")) for item in responses]
    return {
        "mas_has_adrd": int(mean_probability >= 0.5 * scale),
        "mas_adrd_conf": mean_probability,
        "mas_adrd_subtype": max(set(subtypes), key=subtypes.count),
        "parallel_votes": json.dumps(votes),
    }


def generate_branch_outputs(patient_rows, branch, output_path, use_chart_review=True):
    if branch not in {"specialist", "parallel"}:
        raise ValueError("branch must be 'specialist' or 'parallel'.")
    runner = run_specialist_branch if branch == "specialist" else run_parallel_branch
    outputs = []
    for _, row in tqdm(patient_rows.iterrows(), total=len(patient_rows), desc=branch):
        if branch == "specialist":
            outputs.append(runner(row, use_chart_review=use_chart_review))
        else:
            outputs.append(runner(row))
    result = pd.concat(
        [patient_rows.reset_index(drop=True), pd.DataFrame(outputs)],
        axis=1,
    )
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_path, index=False)
    return result


## Method 5 MoE_b (Dual KL Calib)


In [ ]:
def get_custom_xgb_obj_bin_dual_kl(
    specialist_probability,
    parallel_probability,
    alpha=0.05,
    beta_specialist=0.05,
    beta_parallel=0.05,
):
    specialist_probability = np.asarray(specialist_probability, dtype=float)
    parallel_probability = np.asarray(parallel_probability, dtype=float)

    def objective(y_true, raw_margin):
        probability = 1.0 / (1.0 + np.exp(-np.clip(raw_margin, -30.0, 30.0)))
        logistic_hessian = probability * (1.0 - probability)

        gradient_ce = probability - y_true
        hessian_ce = logistic_hessian

        gradient_brier = 2.0 * (probability - y_true) * logistic_hessian
        hessian_brier = 2.0 * logistic_hessian * (
            (probability - y_true) * (1.0 - 2.0 * probability) + logistic_hessian
        )

        gradient_specialist = probability - specialist_probability
        gradient_parallel = probability - parallel_probability

        gradient = (
            gradient_ce
            + alpha * gradient_brier
            + beta_specialist * gradient_specialist
            + beta_parallel * gradient_parallel
        )
        hessian = (
            hessian_ce
            + alpha * hessian_brier
            + beta_specialist * logistic_hessian
            + beta_parallel * logistic_hessian
        )
        return gradient, np.maximum(hessian, 1e-6)

    return objective


def probability_unit_interval(values):
    values = pd.to_numeric(values, errors="raise").to_numpy(dtype=float)
    if np.nanmax(values) > 1.0:
        values = values / 100.0
    if np.any((values < 0.0) | (values > 1.0)):
        raise ValueError("Branch probabilities must lie in [0, 1] or [0, 100].")
    return np.clip(values, 1e-6, 1.0 - 1e-6)


def align_branch_outputs(specialist, parallel, patient_col, label_col):
    required = {patient_col, label_col}
    for name, table in (("specialist", specialist), ("parallel", parallel)):
        missing = required.difference(table.columns)
        if missing:
            raise KeyError(f"{name} branch is missing columns: {sorted(missing)}")
        if table[patient_col].duplicated().any():
            raise ValueError(f"{name} branch contains duplicate patient identifiers.")

    parallel_copy = parallel.copy()
    parallel_copy = parallel_copy.rename(
        columns={
            column: f"parallel_{column}"
            for column in parallel_copy.columns
            if column not in {patient_col, label_col}
        }
    )
    aligned = specialist.merge(
        parallel_copy,
        on=[patient_col, label_col],
        how="inner",
        validate="one_to_one",
    )
    if len(aligned) != len(specialist) or len(aligned) != len(parallel):
        raise ValueError("Specialist and Parallel branch rows are not patient-aligned.")
    return aligned.sort_values(patient_col).reset_index(drop=True)

def make_method5_model(specialist_probability, parallel_probability):
    return xgb.XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        min_child_weight=1.0,
        random_state=GLOBAL_SEED,
        n_jobs=-1,
        eval_metric="logloss",
        objective=get_custom_xgb_obj_bin_dual_kl(
            specialist_probability,
            parallel_probability,
        ),
    )


def cross_validate_method5_moe_b(
    train_table,
    patient_col,
    label_col,
    specialist_probability_col,
    parallel_probability_col,
    model_feature_columns,
):
    features, feature_columns = numeric_feature_matrix(
        train_table,
        model_feature_columns,
        patient_col=patient_col,
        label_col=label_col,
    )
    labels = train_table[label_col].astype(int).to_numpy()
    groups = train_table[patient_col].to_numpy()
    folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=GLOBAL_SEED)
    out_of_fold = np.zeros(len(train_table), dtype=float)

    for train_index, validation_index in folds.split(features, labels, groups):
        specialist_probability = probability_unit_interval(
            train_table.iloc[train_index][specialist_probability_col]
        )
        parallel_probability = probability_unit_interval(
            train_table.iloc[train_index][parallel_probability_col]
        )
        model = make_method5_model(specialist_probability, parallel_probability)
        model.fit(features.iloc[train_index], labels[train_index])
        out_of_fold[validation_index] = model.predict_proba(features.iloc[validation_index])[:, 1]

    return {
        "feature_columns": feature_columns,
        "feature_audit": features.attrs["feature_audit"],
        "probability": out_of_fold,
        "auc": roc_auc_score(labels, out_of_fold),
        "brier": brier_score_loss(labels, out_of_fold),
    }


def fit_and_evaluate_method5_moe_b(
    train_table,
    test_table,
    patient_col,
    label_col,
    specialist_probability_col,
    parallel_probability_col,
    model_feature_columns,
):
    overlap = set(train_table[patient_col]).intersection(set(test_table[patient_col]))
    if overlap:
        raise ValueError("Patient overlap detected between training and test sets.")

    train_features, feature_columns = numeric_feature_matrix(
        train_table,
        model_feature_columns,
        patient_col=patient_col,
        label_col=label_col,
    )
    test_features, _ = numeric_feature_matrix(
        test_table,
        feature_columns,
        patient_col=patient_col,
        label_col=label_col,
        imputation_values=train_features.median(numeric_only=True),
    )

    train_labels = train_table[label_col].astype(int).to_numpy()
    test_labels = test_table[label_col].astype(int).to_numpy()
    specialist_probability = probability_unit_interval(
        train_table[specialist_probability_col]
    )
    parallel_probability = probability_unit_interval(
        train_table[parallel_probability_col]
    )

    model = make_method5_model(specialist_probability, parallel_probability)
    model.fit(train_features, train_labels)
    probability = model.predict_proba(test_features)[:, 1]
    prediction = (probability >= 0.5).astype(int)
    metrics = {
        "accuracy": accuracy_score(test_labels, prediction),
        "f1": f1_score(test_labels, prediction),
        "auc": roc_auc_score(test_labels, probability),
        "brier": brier_score_loss(test_labels, probability),
        "feature_columns": feature_columns,
        "feature_audit": train_features.attrs["feature_audit"],
    }
    return model, metrics, pd.DataFrame(
        {
            patient_col: test_table[patient_col].to_numpy(),
            "label": test_labels,
            "prediction": prediction,
            "probability": probability,
        }
    )


## Patient-aligned execution


In [ ]:
PATIENT_COLUMN = "subject_id"
LABEL_COLUMN = "label_numerical"
SPECIALIST_PROBABILITY_COLUMN = "mas_adrd_conf"
PARALLEL_PROBABILITY_COLUMN = "parallel_mas_adrd_conf"
MODEL_FEATURE_COLUMNS = (
    # Structured covariates available at the prediction index.
    "anchor_age",
    # Numeric Specialist-branch features defined by the prompts above.
    "feat_symptom_symptoms_list_count",
    "feat_symptom_has_behavioral_disturbance",
    "feat_symptom_has_gait_or_falls",
    "feat_symptom_has_memory_complaint",
    "feat_medication_medications_count",
    "feat_risk_imaging_findings_count",
    "feat_risk_risks_history_count",
    # Continuous branch probabilities; hard predictions are excluded.
    SPECIALIST_PROBABILITY_COLUMN,
    PARALLEL_PROBABILITY_COLUMN,
)


def run_method5_from_csv(
    specialist_train_path,
    specialist_test_path,
    parallel_train_path,
    parallel_test_path,
):
    specialist_train = pd.read_csv(specialist_train_path)
    specialist_test = pd.read_csv(specialist_test_path)
    parallel_train = pd.read_csv(parallel_train_path)
    parallel_test = pd.read_csv(parallel_test_path)

    train_table = align_branch_outputs(
        specialist_train,
        parallel_train,
        PATIENT_COLUMN,
        LABEL_COLUMN,
    )
    test_table = align_branch_outputs(
        specialist_test,
        parallel_test,
        PATIENT_COLUMN,
        LABEL_COLUMN,
    )
    cv_metrics = cross_validate_method5_moe_b(
        train_table,
        PATIENT_COLUMN,
        LABEL_COLUMN,
        SPECIALIST_PROBABILITY_COLUMN,
        PARALLEL_PROBABILITY_COLUMN,
        MODEL_FEATURE_COLUMNS,
    )
    model, test_metrics, predictions = fit_and_evaluate_method5_moe_b(
        train_table,
        test_table,
        PATIENT_COLUMN,
        LABEL_COLUMN,
        SPECIALIST_PROBABILITY_COLUMN,
        PARALLEL_PROBABILITY_COLUMN,
        MODEL_FEATURE_COLUMNS,
    )
    return model, cv_metrics, test_metrics, predictions


# Supply the four CSV paths and call run_method5_from_csv.
